In [1]:
# !pip install transformers torch pandas matplotlib peft

In [1]:
import json
import math
import random
import torch



import os

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm


import matplotlib.pyplot as plt
import pandas as pd


In [2]:
from string import Template
import string

prompt_template= Template('''
$question                                           
$options
''')


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("allenai/unifiedqa-v2-t5-3b-1363200")
model = AutoModelForSeq2SeqLM.from_pretrained("allenai/unifiedqa-v2-t5-3b-1363200", device_map="auto")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [38]:
# Function to clean and split text into words
def preprocess(text):
    
    text = text.lower()
    # return set(text.split())
    cleaned = []
    for word in text.split():
        if word.endswith("."):
            word= word[:-1]
        cleaned.append(word)
    return set(cleaned)
            
    


def choose_most_likely_option(options, candidate_answer):
    candidate_words = preprocess(candidate_answer)

    best_option = None
    max_overlap = 0
    
    for option in options:
        option_words = preprocess(option)
        overlap = len(candidate_words.intersection(option_words))


        if overlap > max_overlap:
            max_overlap = overlap
            best_option = option


    if  not best_option:
        
        best_option = None
        max_overlap = 0
        new_candidate = []
        for z in candidate_words :
            new_candidate.extend( list(z))
        for option in options:
            option_words = preprocess(option)

            new_options =[]
            for z in option_words :
                new_options.extend( list(z))

            overlap = len( set(list(new_candidate)).intersection( set(list(new_candidate))) )

            if overlap > max_overlap:
                max_overlap = overlap
                best_option = option

        return best_option, options.index(best_option)+1

            
    return best_option, options.index(best_option)+1

In [55]:
class MyLLMDataloader:
    def __init__(self, batch_size, tokenizer, data, shuffle = False, val= False):
        ## initializations
        self.batch_size  = batch_size
        self.tokenizer  = tokenizer
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.data = []
        self.shuffle = shuffle
        self.val = val
        with open(data, "r") as f:
            test_data = f.readlines()
        if self.val:
            for line in test_data:
                self.data.append(json.loads(line))
        else:
             for line in test_data[:30000]:
                self.data.append(json.loads(line))              
        self.n_data_points = math.ceil(len(self.data)/self.batch_size)
        self.indices = [i for i in range(self.n_data_points)]
        
        print("datapoints",self.n_data_points , len(self.data))
        
    def __getitem__(self, idx):
        ## this gets a batch 
        global answerout
        global promptout
        option_header = ["option A ", "option B ", "option C ", "option D ", "option E "]

        mapper_ans = {"A":1, "B":2, "C":3, "D":4, "E":5, "F":6, "G":7, "H":8}

        batch_start_id = idx * self.batch_size
        batch_end_id  = min(len(self.data), batch_start_id + self.batch_size) 
        batch = {"question_context":[], "answer":[], "options": [], "answer_option":[]}
        mapper_ans_inverted = {1:"a", 2:"b", 3:"c", 4:"d", 5:"e"}
        
        for i in range(batch_start_id, batch_end_id):
            example = self.data[i]
            options = []
            for key in example.keys():
                if key.startswith("op"):
                    options.append(example[key])

            correct_option_id = example["cop"]
            correct_option_txt =example["op"+mapper_ans_inverted[correct_option_id]]
            explanation = ''
            if example["exp"] != None:
                explanation  = example["exp"]


    

            if self.shuffle:
                random.shuffle(options)
                correct_option_id = options.index(correct_option_txt) +1
            
                options_with_header = [option_header[i] +options[i] for i in range(len(options)) ]
                options_with_header = "\n".join(options_with_header)
                # correct_option_txt = example["answer"].split(": ")[1]
                # correct_option_txt = example["answer"][10:]

                # correct_option_txt_header = mapper_ans_inverted[correct_option_id].upper() +" " + correct_option_txt
                correct_option_txt_header =  correct_option_txt

                prompt = prompt_template.substitute(question = example["question"],options =options_with_header)

            else:
                correct_option_id = correct_option_id
                # correct_option_txt_header = mapper_ans_inverted[correct_option_id].upper() +" " + correct_option_txt
                correct_option_txt_header =  correct_option_txt

                options_with_header = [option_header[i] +options[i] for i in range(len(options)) ]
                options_with_header = "\n".join(options_with_header)
                prompt = prompt_template.substitute(question = example["question"] , options =options_with_header)
            ## context TBD
            # answer =  f"{correct_option_txt_header}  \nExplanation: {example['explanation']}"
            if not self.val:
                answer =  f"{correct_option_txt_header}  \nExplanation: {explanation}"
            else:
                answer =  f"{correct_option_txt_header}"

         
            correct_option_txt_header += '\n' + 'explanation: ' + explanation


        
            batch["question_context"] += [prompt]

            batch["answer"] += [correct_option_txt_header]
            batch["options"].append(options)
            batch["answer_option"] += [correct_option_txt]
        
        print(prompt,batch["answer_option"])
        # self.tokenizer.padding_side = "left"
        q_tokens = self.tokenizer(batch["question_context"], padding="longest", max_length= 512, truncation=True, return_tensors="pt")  
        # self.tokenizer.padding_side = "right"
        a_tokens = self.tokenizer(batch["answer"], padding="longest", max_length= 128, truncation=True, return_tensors="pt")
        tokens = torch.cat([q_tokens["input_ids"], a_tokens["input_ids"]], dim=1)
        attn_masks = torch.cat([q_tokens["attention_mask"], a_tokens["attention_mask"]], dim=1)
        loss_mask = torch.cat([torch.zeros_like(q_tokens["attention_mask"]), a_tokens["attention_mask"]], dim=1)[:,1:]
        

        result = {
        "question_context":q_tokens,
        'question_text' : batch["question_context"],
        'answer_text' : batch["answer"],
        'options': batch["options"],
        "a_tokens":a_tokens,## Causal Training
        'answer_option': batch["answer_option"]
        }
        # result["loss_mask"] = loss_mask * result["out_mask"]
        # result["out_ids"][:,:q_tokens["input_ids"].size(1)-10] = self.tokenizer.eos_token_id

        return result       


            

    def __iter__(self):
        self.idx = 0
        return self

    def __next__(self):
        if self.idx >= self.n_data_points:
            self.idx = 0
            raise StopIteration
        temp_idx = self.indices[self.idx]
        self.idx += 1
        return self[temp_idx]
                
    def __len__(self):
        return self.n_data_points
    
     

In [56]:
trainLoader = MyLLMDataloader(2, tokenizer, "dev.json", val=False, shuffle=True)
for z in trainLoader:
    break

datapoints 2092 4183

Which of the following is not true about glomerular capillaries')                                           
option A Glucose concentration in the capillaries is the same as that in glomerular filtrate
option B The oncotic pressure of the fluid leaving the capillaries is less than that of fluid entering it
option C Constriction of afferent aeriole decreases the blood flow to the glomerulas
option D Hematocrit of the fluid leaving the capillaries is less than that of the fluid entering it
 ['Impulse through myelinated fibers is slower than non-myelinated fibers', 'The oncotic pressure of the fluid leaving the capillaries is less than that of fluid entering it']


In [57]:
def forward_pass(model, batch):
    inp_ids = batch["question_context"]["input_ids"].to(model.device)
    attn_mask = batch["question_context"]["attention_mask"].to(model.device)

    a_tokens = batch['a_tokens']['input_ids'].to(model.device)
    # print(inp_ids.shape, attn_mask.shape, a_tokens.shape)
    # print(inp_ids[0][:10], attn_mask[0][:10], a_tokens[0][:10])
    a_tokens[a_tokens == tokenizer.pad_token_id] = -100
    # result = model(input_ids=inp_ids, attention_mask=attn_mask)


    result = model(input_ids=inp_ids,attention_mask =attn_mask, labels = a_tokens)
    logits = result.logits
    # print(result.loss)
    # forward_pass_2(batch["question_text"],batch["answer_text"] )
    return result.loss, logits

def forward_pass_2(input_sequences, output_sequences):
        # the following 2 hyperparameters are task-specific
    max_source_length = 512
    max_target_length = 128
    print("inp_seq", input_sequences)
    print("out_seq", output_sequences)


    # input_sequences = [promptout]

    encoding = tokenizer(
        input_sequences,
        padding="longest",
        max_length=max_source_length,
        truncation=True,
        return_tensors="pt",
    ).to("cuda")
    # print(encod)
    input_ids, attention_mask = encoding.input_ids, encoding.attention_mask
    # encode the targets
    target_encoding = tokenizer(
        output_sequences,
        padding="longest",
        max_length=max_target_length,
        truncation=True,
        return_tensors="pt",
    ).to("cuda")
    labels = target_encoding.input_ids

    # replace padding token id's of the labels by -100 so it's ignored by the loss
    labels[labels == tokenizer.pad_token_id] = -100



    # forward pass
    res = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    
    print("fp2", res.loss.item())

def calc_loss(loss_fn, logits, batch):
    B, L, C = logits.shape
    target = batch["out_ids"].to(logits.device)
    mask = batch["loss_mask"].to(logits.device)
    loss = loss_fn(logits.reshape(-1, C), target.reshape(-1)) * mask.reshape(-1)
    loss = loss.sum()/mask.sum()
    return loss

def update(model, optimizer, loss_fn, batch, accumulate_grad=True):
   
    loss, logits = forward_pass(model, batch)
        # loss = calc_loss(loss_fn, logits, batch)
    
    scaler.scale(loss).backward()
    if not accumulate_grad:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
    return loss.item(), logits




def train_on_batches(model, train_data_loader, optimizer, loss_fn, grad_accum_bs=8):
    train_loss = 0
    score = 0
    num_correct = 0
    total_num = 0
    mapper_ans = {"A":1, "B":2, "C":3, "D":4, "E":5}

    pbar = tqdm(range(len(train_data_loader)))
    option_ids = [tokenizer(o).input_ids[0] for o in ["1", "2", "3", "4", "5","6", "7", "8"]]
    model.train()
    for i,batch in enumerate(train_data_loader):
        grad_step_bs = grad_accum_bs/train_data_loader.batch_size
        if (i>=grad_step_bs and i%grad_step_bs == 0) or i == len(train_data_loader)-1:
            accumulate_grad = False
        else:
            accumulate_grad = True
        loss, logits = update(model, optimizer, loss_fn, batch, accumulate_grad=accumulate_grad)
        train_loss += (1/(i+1))*(loss-train_loss)

        # pred = (logits[:,batch["a_tokens"].input_ids.size(1)-1,option_ids].argmax(dim=1) + 1).tolist()
            

        pred = train_data_loader.tokenizer.batch_decode(logits.argmax(dim=2), skip_special_tokens=True)
        target = train_data_loader.tokenizer.batch_decode(batch["a_tokens"].input_ids[:], skip_special_tokens=True)
        for ops, pre,t in zip(batch["options"], pred, batch["answer_option"]):

            ans, p = choose_most_likely_option(ops, pre)
            print(t, ops, pre)
            total_num += 1


            num_correct += (int(p) == int(ops.index( t)+1))
            score = num_correct/total_num
        

        pbar.set_description(f"Train Loss: {train_loss:.4f} Score: {score:.4f}")
        pbar.update(1)
    pbar.close()


    return train_loss, score



In [58]:
def train(model, tdataloader,vdataloader, optimizer, loss_fn, scheduler, epochs=3, log_dir="logs/t5_medcq"):
    os.makedirs(log_dir, exist_ok=True)
    with open(f"{log_dir}/train.txt", "w") as tf, open(f"{log_dir}/val.txt", "w") as vf:
        tf.write(f"Epoch,Loss,Score\n")
        vf.write("Epoch,Loss,Score\n")
    best_val_score = 0
    
    for epoch in range(1,1+epochs):
        train_loss, tscore = train_on_batches(model, tdataloader, optimizer, loss_fn, grad_accum_bs=16)
        scheduler.step()
        val_loss, score = validation(model, vdataloader, loss_fn)
        with open(f"{log_dir}/train.txt", "a+") as tf, open(f"{log_dir}/val.txt", "a+") as vf:
            tf.write(f"{epoch},{train_loss},{tscore}\n")
            vf.write(f"{epoch},{val_loss},{score}\n")
        lr = optimizer.param_groups[0]["lr"]
        tqdm.write(f"Epoch: {epoch} | LR: {lr:.7f} | Train Loss: {train_loss:.4f} | Train Score: {tscore:.4f} | Val Loss: {val_loss:.4f} | Val Score: {score:.4f}")
        
        if score > best_val_score:
            model.save_pretrained(log_dir+"/model")
            best_val_score = score

        # update the learning rate
        # for param_group in optimizer.param_groups:
        #     param_group['lr'] = 1.0*float(param_group['lr'])

        train_df = pd.read_csv(f"{log_dir}/train.txt")
        val_df = pd.read_csv(f"{log_dir}/val.txt")

        fig, ax = plt.subplots(1, 2, figsize=(20,8))
        ax[0].plot(range(1,len(train_df)+1), train_df["Loss"], label="Train")
        ax[0].plot(range(1,len(val_df)+1), val_df["Loss"], label="Val")

        ax[1].plot(range(1,len(train_df)+1), train_df["Score"], label="Train")
        ax[1].plot(range(1,len(val_df)+1), val_df["Score"], label="Val")

        ax[0].set_xlabel("Epochs")
        ax[1].set_xlabel("Epochs")

        ax[0].set_ylabel("Loss")
        ax[1].set_ylabel("Score")

        ax[0].legend()
        ax[1].legend()

        fig.suptitle('', fontsize=16)
        fig.savefig("plt.png")


def validation(model, val_data_loader, loss_fn):
    val_loss = 0
    mapper_ans = {"A":1, "B":2, "C":3, "D":4, "E":5}

    score = 0
    num_correct = 0
    pbar = tqdm(range(len(val_data_loader)))
    option_ids = [tokenizer(o).input_ids[0] for o in ["1", "2", "3", "4", "5"]]
    total_num = 0
    model.eval()
    with torch.inference_mode():
        for i, batch in enumerate(val_data_loader):

            # loss,logits = forward_pass(model, batch)
   
            # val_loss += (1/(i+1))*(loss - val_loss)
            # pred = (logits[:,batch["a_tokens"].input_ids.size(1)-1,option_ids].argmax(dim=1) + 1).tolist()
            gen_tokens = model.generate(**batch["question_context"].to(model.device), max_new_tokens=10)
       
            pred = val_data_loader.tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)

            target = val_data_loader.tokenizer.batch_decode(batch["a_tokens"].input_ids[:,0], skip_special_tokens=True)

            for ops, pre,t in zip(batch["options"], pred, batch["answer_option"]):
                ans, p = choose_most_likely_option(ops, pre)
   
                total_num += 1
                # print('p', p, 't',t)


                # num_correct += (int(p) == int(t))
                num_correct += (int(p) == int(ops.index( t)+1))

                score = num_correct/total_num
                
            pbar.set_description(f"Val Loss: {val_loss:.4f} Score: {score:.4f}")
            pbar.update(1)
            # break
        gen_tokens = model.generate(**batch["question_context"].to(model.device), max_new_tokens=10)
        gen_txt = val_data_loader.tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        target_txt = val_data_loader.tokenizer.batch_decode(batch["a_tokens"].input_ids, skip_special_tokens=True)
        tqdm.write("Target: " + "\n" + "\n".join(target_txt) + "\nGenerated: \n " + "\n".join(gen_txt) +"\n")
        pbar.close()

    return val_loss, score


In [59]:
# device = "cuda"
# torch.set_default_device(device)
# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, device_map ="auto")
epochs = 10
lr = 1e-4

loss_fn = torch.nn.CrossEntropyLoss(ignore_index=tokenizer.eos_token_id, reduction='none').cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr, weight_decay= 0.001)
optimizer.zero_grad()


# trainLoader = TrainDataLoader(2, tokenizer, topk=topk)
trainLoader = MyLLMDataloader(8, tokenizer, "train5000_cleaned.json", val=False, shuffle=True)

valLoader = MyLLMDataloader(1, tokenizer, "dev.json", val=True)
scaler =  torch.cuda.amp.GradScaler(enabled=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,
            T_max = epochs*2, eta_min=1E-8)
train(model, trainLoader, valLoader,optimizer, loss_fn, scheduler, epochs=10)



datapoints 625 5000
datapoints 4183 4183


/tmp/ipykernel_292804/3333636131.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler =  torch.cuda.amp.GradScaler(enabled=True)


  0%|          | 0/625 [00:00<?, ?it/s]


Per rectum examination is not a useful test for diagnosis of                                           
option A Hemorrhoid
option B Pilonidal sinus
option C Rectal ulcer
option D Anal fissure
 ['Atrophy', 'Vitamin B12', 'Roux en Y Duodenal By pass', 'Central aery of the retina', 'IG1-1', 'Mite', 'Satellite lesions', 'Pilonidal sinus']
Atrophy ['Atrophy', 'Hyperophy', 'Dyplasia', 'Hyperplasia'] hyperrophy option option urethral obstruction due of benigny us hyperstatic la, benign,, cyst cyst syndrome hypers, cystnine lapse, cyst cyst renalpeurisis causes  means  to describe kidneyatation of the tubs urinari formation with renal renalrophy. renal renal par to benign of  flow  urineredersr 7sy  299d hyper., hyper hyper hyper hyper hyper hyper hyper hyper hyper hyper hyper hyper hyper hyper hyper
Vitamin B12 ['Vitamin D', 'Vitamin B7', 'Vitamin B12', 'Vitamin C'] vitamin D12 D12actionuse clause's 05 amendment&dict.,.a, D12appermin) esized fromly in animalorganisms human, it vitamin anima

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 47.71 GiB of which 42.44 MiB is free. Including non-PyTorch memory, this process has 42.66 GiB memory in use. Of the allocated memory 41.94 GiB is allocated by PyTorch, and 234.10 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)